# Terminologia e Implementacoes em Python — Teoria dos Grafos

Este notebook reescreve e implementa em Python o conteúdo do documento "01 - Terminologia".
Contém definições, representações, algoritmos e exemplos práticos usando Python e NetworkX.

## Índice

- [Introdução e definições](#Introducao-e-definicoes)
- [Terminologia essencial](#Terminologia-essencial)
- [Representações e implementações básicas](#Representacoes-e-implementacoes-basicas)
- [Grafo simples, completo, ciclo, roda, bipartido](#Tipos-de-grafos-com-exemplos)
- [Grau e o Teorema do Aperto de Mãos (Handshake)](#Grau-e-teorema-do-aperto-de-maos)
- [Havel–Hakimi: existência de sequência de graus](#Havel-Hakimi)
- [Grafo dirigido e propriedades (in/out-degree)](#Grafos-dirigidos)
- [Busca: BFS e DFS (implementações)](#BFS-e-DFS)
- [Caminhos mínimos: Dijkstra e Bellman-Ford (exemplos)](#Caminhos-minimos)
- [Eulerianidade e o problema de Königsberg](#Eulerianidade-e-Konigsberg)
- [Exemplos e visualizações com NetworkX](#Exemplos-e-visualizacoes)
- [Referências e exercícios sugeridos](#Referencias-e-exercicios)

## Introducao e definicoes

Um grafo $G$ é um par $G=(V,E)$ onde $V$ é o conjunto de vértices (ou nós) e $E$ é um conjunto de arestas. Cada aresta conecta um subconjunto de vértices (normalmente um par).

Usos: redes de computadores, redes sociais, modelagem de problemas de otimização, representações químicas, etc.

## Terminologia essencial

- Vértice (nó): elemento de `V`.
- Aresta: elemento de `E`. Em grafos não direcionados, uma aresta é um par não ordenado {u, v}; em grafos direcionados é um par ordenado (u, v).
- Laço (loop): aresta que liga um vértice a si mesmo.
- Arestas paralelas: múltiplas arestas entre os mesmos vértices (multigrafo).
- Adjacência: dois vértices são adjacentes se existe uma aresta entre eles.
- Grau: número de arestas incidentes em um vértice (laços contam duas vezes).
- Caminho, ciclo, componente conexo, diâmetro, raio, subgrafo, grafo completo, grafo bipartido, árvore, etc.

## Representacoes e implementacoes basicas

As representações mais comuns são: lista de adjacência, matriz de adjacência e lista de arestas. Abaixo implementamos estruturas simples em Python e operações básicas.

In [ ]:
# Representacao: Lista de adjacencia (dicionario de listas)
class AdjListGraph:
    def __init__(self, directed=False):
        self.adj = {}
        self.directed = directed

    def add_vertex(self, v):
        if v not in self.adj:
            self.adj[v] = []

    def add_edge(self, u, v):
        self.add_vertex(u)
        self.add_vertex(v)
        self.adj[u].append(v)
        if not self.directed:
            self.adj[v].append(u)

    def vertices(self):
        return list(self.adj.keys())

    def edges(self):
        es = []
        for u, nbrs in self.adj.items():
            for v in nbrs:
                if self.directed or u <= v:
                    es.append((u, v))
        return es

    def degree(self, v):
        if self.directed:
            raise ValueError('use indegree/outdegree para grafos dirigidos')
        return len(self.adj.get(v, []))

    def __repr__(self):
        return f'AdjListGraph(directed={self.directed}, V={len(self.adj)}, E={len(self.edges())})'

# Matriz de adjacencia (lista de listas)
class AdjMatrixGraph:
    def __init__(self, n_vertices, directed=False):
        self.n = n_vertices
        self.directed = directed
        self.mat = [[0]*n_vertices for _ in range(n_vertices)]

    def add_edge(self, u, v, weight=1):
        self.mat[u][v] = weight
        if not self.directed:
            self.mat[v][u] = weight

    def adjacency(self, u):
        return [i for i,w in enumerate(self.mat[u]) if w != 0]

    def __repr__(self):
        return f'AdjMatrixGraph(n={self.n}, directed={self.directed})'

# Exemplo rapido
g = AdjListGraph()
g.add_edge('A','B')
g.add_edge('A','C')
g.add_edge('B','C')
print(g)
print('Vertices:', g.vertices())
print('Arestas:', g.edges())
print('Grau de A:', g.degree('A'))

## Tipos de grafos (com exemplos gerados em NetworkX)

Vamos gerar grafos completos, ciclos, rodas e bipartidos para exemplificar.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Grafo completo K_n
def show_complete(n, ax=None):
    G = nx.complete_graph(n)
    if ax is None:
        plt.figure()
    nx.draw_circular(G, with_labels=True, node_color='lightblue')
    plt.title(f'K_{n} - grafo completo')
    plt.show()

show_complete(5)
# Ciclo C_n
plt.figure()
C = nx.cycle_graph(6)
nx.draw_circular(C, with_labels=True, node_color='lightgreen')
plt.title('C_6 - ciclo')
plt.show()
# Roda W_n (cycle + central)
plt.figure()
W = nx.wheel_graph(7)
nx.draw_circular(W, with_labels=True, node_color='lightcoral')
plt.title('W_7 - roda')
plt.show()
# Bipartido completo K_{3,4}
plt.figure()
B = nx.complete_bipartite_graph(3,4)
pos = nx.spring_layout(B)
nx.draw(B, pos, with_labels=True, node_color='khaki')
plt.title('K_{3,4} - bipartido completo')
plt.show()

## Grau e o Teorema do Aperto de Mãos (Handshake)

O teorema afirma: a soma dos graus dos vértices é igual a duas vezes o número de arestas: $\sum_v deg(v) = 2|E|$. Vamos demonstrar numericamente em um grafo.

In [ ]:
G = nx.path_graph(7)  # caminho de 7 vértices
degrees = dict(G.degree())
sum_degrees = sum(degrees.values())
num_edges = G.number_of_edges()
print('graus =', degrees)
print('soma dos graus =', sum_degrees)
print('2 * |E| =', 2 * num_edges)

## Havel–Hakimi: verificar se uma sequência de inteiros é sequência de graus de um grafo simples

O algoritmo reduz iterativamente a sequência: ordena, remove o maior d e subtrai 1 dos próximos d elementos; se em algum passo aparecer número negativo ou não for possível, a sequência não é gráfica.

In [ ]:
def havel_hakimi(seq):
    seq = [d for d in seq if d > 0]  # remove zeros
    seq.sort(reverse=True)
    if not seq:
        return True
    d = seq.pop(0)
    if d > len(seq):
        return False
    for i in range(d):
        seq[i] -= 1
        if seq[i] < 0:
            return False
    return havel_hakimi(seq)

# Exemplos
print(havel_hakimi([3,3,2,2,2]))  # True (ex: K_5 minus algumas arestas)
print(havel_hakimi([4,4,4,1]))    # False

## Grafos dirigidos — in-degree e out-degree

Em digrafos, cada aresta tem direção. O in-degree de v é o número de arestas entrando em v; out-degree é o número saindo.
Vamos construir um digrafo e mostrar essas medidas.

In [ ]:
DG = nx.DiGraph()
DG.add_edges_from([(1,2),(2,3),(3,1),(3,4),(5,3)])
print('in_degree:', dict(DG.in_degree()))
print('out_degree:', dict(DG.out_degree()))

## BFS e DFS — implementações e exemplos

Implementaremos BFS (largura) e DFS (profundidade) de forma iterativa/recursiva e mostraremos trajetos e árvores geradas.

In [ ]:
from collections import deque

def bfs(graph_adj, start):
    visited = set([start])
    q = deque([start])
    order = []
    parent = {start: None}
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph_adj.get(u, []):
            if v not in visited:
                visited.add(v)
                parent[v] = u
                q.append(v)
    return order, parent

def dfs_recursive(graph_adj, start, visited=None, order=None, parent=None):
    if visited is None:
        visited = set()
    if order is None:
        order = []
    if parent is None:
        parent = {start: None}
    visited.add(start)
    order.append(start)
    for v in graph_adj.get(start, []):
        if v not in visited:
            parent[v] = start
            dfs_recursive(graph_adj, v, visited, order, parent)
    return order, parent

# Exemplo usando um pequeno grafo
adj = {'A':['B','C'], 'B':['A','D'], 'C':['A','D'], 'D':['B','C','E'], 'E':['D']}
print('BFS from A:', bfs(adj, 'A')[0])
print('DFS from A:', dfs_recursive(adj, 'A')[0])

## Caminhos mínimos — Dijkstra (com heap) e Bellman-Ford (simples)

Mostraremos implementações concisas e testaremos em um pequeno grafo ponderado.

In [ ]:
import heapq
def dijkstra(adj_weighted, source):
    dist = {v: float('inf') for v in adj_weighted}
    dist[source] = 0
    prev = {v: None for v in adj_weighted}
    heap = [(0, source)]
    while heap:
        d,u = heapq.heappop(heap)
        if d>dist[u]:
            continue
        for v,w in adj_weighted[u].items():
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(heap, (nd, v))
    return dist, prev

# Bellman-Ford (detectar arestas negativas)
def bellman_ford(adj_weighted, source):
    dist = {v: float('inf') for v in adj_weighted}
    dist[source] = 0
    prev = {v: None for v in adj_weighted}
    V = list(adj_weighted.keys())
    for _ in range(len(V)-1):
        updated = False
        for u in adj_weighted:
            for v,w in adj_weighted[u].items():
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    prev[v] = u
                    updated = True
        if not updated:
            break
    # checar ciclo negativo
    for u in adj_weighted:
        for v,w in adj_weighted[u].items():
            if dist[u] + w < dist[v]:
                raise ValueError('Ciclo de peso negativo detectado')
    return dist, prev

# Exemplo
adjw = {
    's': {'a':1,'b':4},
    'a': {'b':2,'c':5},
    'b': {'c':1},
    'c': {}
}
print('Dijkstra from s:', dijkstra(adjw,'s')[0])
print('Bellman-Ford from s:', bellman_ford(adjw,'s')[0])

## Eulerianidade e o problema de Königsberg

Um grafo (não direcionado) possui uma trilha euleriana que percorre cada aresta exatamente uma vez se e somente se for conexo (ignorando vértices isolados) e tiver 0 ou 2 vértices de grau ímpar (0 -> ciclo euleriano, 2 -> trilha aberta).

Vamos modelar o problema das sete pontes e verificar se há solução.

In [ ]:
# Exemplo de Konigsberg (modelo simples): quatro regiao A,B,C,D com 7 pontes entre elas
G = nx.Graph()
# Um exemplo simplificado que representa as sete pontes (não é unica modelagem)
G.add_edges_from([('A','B'),('A','B'),('A','C'),('A','D'),('B','C'),('B','D'),('C','D')])
print('graus:', dict(G.degree()))
odd = [v for v,d in G.degree() if d%2==1]
print('vertices de grau impar:', odd)
print('Existe trilha euleriana? (0 ou 2 ímpares):', len(odd) in (0,2))
# Resultado: no caso das sete pontes, mais de dois vértices têm grau ímpar, logo nao ha trilha euleriana.

## Exemplos e visualizacoes

Aqui há código para criar grafos a partir de sequências de graus, visualizar grafos e experimentar com NetworkX.

In [ ]:
# Criar grafo a partir de uma sequência valida de graus (construcao gulosa simples usando Havel-Hakimi)
def construct_from_degree_sequence(seq):
    # seq: lista de graus (assume que é grafica) - algoritmo construtivo simples
    import copy
    seq = list(seq)
    n = len(seq)
    # rotular vertices como 0..n-1
    vertices = list(range(n))
    # pares (deg, vertex)
    pairs = sorted([(seq[i], i) for i in range(n)], reverse=True)
    G = nx.Graph()
    G.add_nodes_from(vertices)
    while pairs and pairs[0][0] > 0:
        d, v = pairs.pop(0)
        if d > len(pairs):
            raise ValueError('sequencia nao grafica')
        for i in range(d):
            pairs[i] = (pairs[i][0]-1, pairs[i][1])
            G.add_edge(v, pairs[i][1])
        pairs = sorted(pairs, reverse=True)
    return G

seq = [3,3,2,2,2]
print('Havel-Hakimi says graphic?', havel_hakimi(seq))
Gseq = construct_from_degree_sequence(seq)
print('G from seq: nodes', Gseq.nodes(), 'edges', Gseq.edges())
plt.figure()
nx.draw_spring(Gseq, with_labels=True, node_color='lightblue')
plt.title('Grafo construído a partir de sequência de graus')
plt.show()

## Referências e exercícios

Referências recomendadas:
- Cormen, Leiserson, Rivest, Stein — Introduction to Algorithms (capítulo de grafos)
- Diestel — Graph Theory
- Gross & Yellen — Graph Theory and Its Applications

Exercícios sugeridos:
1. Aplicar Havel–Hakimi em diversas sequências aleatórias e construir grafos quando possível.
2. Implementar e testar algoritmo de Kruskal para MST e comparar com NetworkX.
3. Resolver variações do problema de Königsberg adicionando/removendo pontes para tornar o grafo euleriano.

---
Notebook gerado em atendimento à solicitação; edite e expanda conforme necessário.